# bsky-pulse — Data Exploration & Sentiment Visualization

This notebook loads collected Bluesky posts and produces:
- **Keyword frequency bar chart** — how often each tracked keyword appeared
- **Word cloud** — most common words in matched posts
- **VADER sentiment distribution** — positive / neutral / negative breakdown
- **Sentiment over time** — rolling compound score

**Before running:** copy `.env.example` → `.env` and fill in your credentials,
or set the values directly in the *Configuration* cell below.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('..') / '.env')  # load project-root .env

STORAGE_BACKEND = os.getenv('STORAGE_BACKEND', 'json')   # 'json' or 'mongodb'
JSON_PATH       = Path(os.getenv('JSON_PATH', '../data/posts.json'))
MONGO_URI       = os.getenv('MONGO_URI', '')
CERT_FILE_PATH  = os.getenv('CERT_FILE_PATH', '')
ROLLING_WINDOW  = '10min'  # adjust to '1h', '30min', etc.

print(f'Backend: {STORAGE_BACKEND}')

In [ ]:
# ── Load posts ───────────────────────────────────────────────────────────────
import json
import pandas as pd
import certifi

def load_posts():
    if STORAGE_BACKEND == 'json':
        if not JSON_PATH.exists():
            raise FileNotFoundError(f'JSON store not found: {JSON_PATH}')
        with JSON_PATH.open('r', encoding='utf-8') as fh:
            return json.load(fh)
    else:
        from pymongo import MongoClient
        if not MONGO_URI:
            raise ValueError('MONGO_URI is not set in your .env file.')
        kwargs = {'tls': True, 'tlsCAFile': certifi.where()}
        if CERT_FILE_PATH:
            kwargs['tlsCertificateKeyFile'] = CERT_FILE_PATH
        client = MongoClient(MONGO_URI, **kwargs)
        posts = list(client['bluesky_research']['posts'].find({}, {'_id': 0}))
        client.close()
        return posts

raw = load_posts()
df = pd.DataFrame(raw)
if 'raw' in df.columns:
    df = df.drop(columns=['raw'])
if 'collected_at' in df.columns:
    df['collected_at'] = pd.to_datetime(df['collected_at'], utc=True)

print(f'Loaded {len(df)} posts')
df.head()

In [ ]:
# ── Keyword frequency ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

kw_counts = df['matched_keyword'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
kw_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Posts per keyword', fontsize=14)
ax.set_xlabel('Keyword')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../data/keyword_frequency.png', dpi=150)
plt.show()

In [ ]:
# ── Word cloud ───────────────────────────────────────────────────────────────
import re
from wordcloud import WordCloud, STOPWORDS

all_text = ' '.join(df['text'].dropna().astype(str).tolist())
# Remove URLs
all_text = re.sub(r'https?://\S+', '', all_text)

wc = WordCloud(
    width=1200,
    height=600,
    background_color='white',
    stopwords=STOPWORDS,
    max_words=150,
    colormap='plasma',
).generate(all_text)

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Word Cloud — bsky-pulse', fontsize=16)
plt.tight_layout()
plt.savefig('../data/wordcloud.png', dpi=150)
plt.show()

In [ ]:
# ── VADER sentiment analysis ─────────────────────────────────────────────────
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def score(text):
    scores = analyzer.polarity_scores(str(text))
    return scores['compound']

def label(compound):
    if compound >= 0.05:
        return 'positive'
    elif compound <= -0.05:
        return 'negative'
    return 'neutral'

df['compound'] = df['text'].apply(score)
df['sentiment'] = df['compound'].apply(label)

print(df['sentiment'].value_counts())
df[['text', 'matched_keyword', 'compound', 'sentiment']].head(10)

In [ ]:
# ── Sentiment distribution pie chart ─────────────────────────────────────────
counts = df['sentiment'].value_counts()
colors = {'positive': '#4caf50', 'neutral': '#90a4ae', 'negative': '#ef5350'}

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    counts,
    labels=counts.index,
    autopct='%1.1f%%',
    colors=[colors.get(s, 'grey') for s in counts.index],
    startangle=140,
)
ax.set_title('Sentiment Distribution', fontsize=14)
plt.tight_layout()
plt.savefig('../data/sentiment_pie.png', dpi=150)
plt.show()

In [ ]:
# ── Sentiment over time (rolling average) ────────────────────────────────────
if 'collected_at' in df.columns and not df['collected_at'].isna().all():
    ts = df.set_index('collected_at')['compound'].sort_index()
    rolling = ts.rolling(ROLLING_WINDOW).mean()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(ts.index, ts.values, alpha=0.3, color='grey', label='raw')
    ax.plot(rolling.index, rolling.values, color='steelblue', lw=2, label=f'{ROLLING_WINDOW} rolling mean')
    ax.axhline(0, color='black', lw=0.8, linestyle='--')
    ax.set_title('Compound Sentiment Score Over Time', fontsize=14)
    ax.set_xlabel('Time (UTC)')
    ax.set_ylabel('VADER Compound Score')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../data/sentiment_over_time.png', dpi=150)
    plt.show()
else:
    print('No timestamp data available — skipping time-series plot.')